## The goal
Build a PyTorch `Dataset` that emits training examples for the two-tower model.

Per `__getitem__` call:
1. One positive interaction `(user_idx, positive_book_idx)` — a book this user rated 4+ stars.
2. **Fresh** sample of 4 negative book indices uniformly drawn from books the user has *not* interacted with at any rating.

Across one epoch:
- The DataLoader iterates from 0 to `len(dataset) - 1`, calling `__getitem__` once per positive.
- Each call generates fresh negatives — the same positive paired with different negatives across epochs.

Key design choices:
- **Anchor on positives.** `len(dataset) = number of positive interactions`. Each item bundles its positive with 4 negatives in a single tuple, keeping the pair coherent.
- **Negatives sampled at runtime, not precomputed.** Zero disk overhead. Fresh per call → effective negative diversity scales with epochs × 4.
- **Exclusion set covers *all* ratings (not just positives).** A book the user rated 2-star shouldn't be sampled as a 'random negative' — it'd be a duplicate signal. A book the user rated 5-star *definitely* shouldn't (label leakage).

Output: a `Dataset` class importable by the training script in step 17. We instantiate and test it here; the actual training loop lives in `mybookrec/model/`.

In [1]:
import json
import numpy as np
import polars as pl
from torch.utils.data import Dataset
from mybookrec import DATA_DIR

In [2]:
class TrainingPairsDataset(Dataset):
    """Emits (user_idx, positive_book_idx, negative_book_indices) per call.

    `__init__` loads positives and per-user exclusion sets once.
    `__getitem__` draws fresh negatives each call via rejection sampling.
    """

    def __init__(self, n_negatives: int = 4, data_split: str = "train", rng_seed: int | None = None):
        # ID mappings (built in earlier steps). user_id_to_index already excludes users with no liked books.
        with open(DATA_DIR / "transformed" / "user_id_to_index.json", "r") as f:
            user_id_to_index = json.load(f)
        with open(DATA_DIR / "transformed" / "book_id_to_index.json", "r") as f:
            book_id_to_index = json.load(f)

        # Stream the 19GB parquet, projecting to the 3 columns we need and filtering to the requested split.
        interactions = (
            pl.scan_parquet(DATA_DIR / "transformed" / "books_with_interactions.parquet")
            .filter(pl.col("data_split") == data_split)
            .select("user_id", "book_id", "rating")
            .collect()
        )

        # Map string ids to integer indices. Anything not in the mappings -> -1, then dropped.
        # The user-side drop also removes users we filtered out in step 13b for having no liked books.
        interactions = (
            interactions
            .with_columns(pl.col("book_id").cast(pl.String))
            .with_columns(
                pl.col("user_id").map_elements(
                    lambda u: user_id_to_index.get(u, -1), return_dtype=pl.Int64
                ).alias("user_idx"),
                pl.col("book_id").map_elements(
                    lambda b: book_id_to_index.get(b, -1), return_dtype=pl.Int64
                ).alias("book_idx"),
            )
            .filter((pl.col("user_idx") >= 0) & (pl.col("book_idx") >= 0))
        )

        # Positives: rating >= 4. These are the training anchors.
        positives = interactions.filter(pl.col("rating") >= 4)
        self.positive_users = positives["user_idx"].to_numpy()
        self.positive_books = positives["book_idx"].to_numpy()

        # Exclusion: every book each user has interacted with, regardless of rating.
        # Stored as numpy arrays — np.isin is fast for the small per-user sets we have (~100 books).
        exclude_groups = (
            interactions.group_by("user_idx").agg(pl.col("book_idx").alias("rated_books"))
        )
        self.exclude = {
            row["user_idx"]: np.array(row["rated_books"], dtype=np.int64)
            for row in exclude_groups.to_dicts()
        }

        self.n_books = len(book_id_to_index)
        self.n_negatives = n_negatives
        self.rng = np.random.default_rng(rng_seed)

        print(f"Positives:           {len(self.positive_users):,}")
        print(f"Users with positives: {len(set(self.positive_users)):,}")
        print(f"Total catalog books: {self.n_books:,}")
        avg_rated = np.mean([len(v) for v in self.exclude.values()])
        print(f"Avg rated books / user: {avg_rated:.1f}")

    def __len__(self) -> int:
        return len(self.positive_users)

    def __getitem__(self, idx: int):
        user_idx = int(self.positive_users[idx])
        pos_book_idx = int(self.positive_books[idx])
        neg_book_indices = self._sample_negatives(user_idx)
        return user_idx, pos_book_idx, neg_book_indices

    def _sample_negatives(self, user_idx: int) -> np.ndarray:
        """Rejection sampling: oversample, drop collisions, top up if needed.

        Collision rate per draw is `len(excluded) / n_books`, typically <0.01%.
        One oversample round is almost always enough.
        """
        excluded = self.exclude.get(user_idx)
        sampled = self.rng.integers(0, self.n_books, size=self.n_negatives * 2)
        if excluded is not None and len(excluded) > 0:
            sampled = sampled[~np.isin(sampled, excluded)]
        while len(sampled) < self.n_negatives:
            extra = self.rng.integers(0, self.n_books, size=self.n_negatives)
            if excluded is not None and len(excluded) > 0:
                extra = extra[~np.isin(extra, excluded)]
            sampled = np.concatenate([sampled, extra])
        return sampled[:self.n_negatives]

In [3]:
# Instantiate. First time this runs the parquet scan + groupby take ~30-60s for ~30-40M train interactions.
dataset = TrainingPairsDataset(n_negatives=4, rng_seed=42)

Positives:           50,971,385
Users with positives: 783,032
Total catalog books: 1,782,579
Avg rated books / user: 92.6


In [4]:
# __len__ check: should equal the number of positive interactions (4+ stars) in the train split.
print(f"Dataset length: {len(dataset):,}")

Dataset length: 50,971,385


In [5]:
# __getitem__ check: pull one sample and inspect its shape.
u, b_pos, b_negs = dataset[0]
print(f"user_idx        = {u}   (scalar int)")
print(f"pos_book_idx    = {b_pos}   (scalar int)")
print(f"neg_book_idxs   = {b_negs}   shape={b_negs.shape}   dtype={b_negs.dtype}")

user_idx        = 428240   (scalar int)
pos_book_idx    = 651306   (scalar int)
neg_book_idxs   = [ 159096 1379637 1166825  782335]   shape=(4,)   dtype=int64


In [6]:
# Fresh negatives demonstration: two consecutive calls to the same index return different negatives.
# This is the resampling-each-epoch property in microcosm.
_, _, negs_a = dataset[0]
_, _, negs_b = dataset[0]
print(f"Call 1 negatives: {negs_a}")
print(f"Call 2 negatives: {negs_b}")
print(f"Identical? {np.array_equal(negs_a, negs_b)}   (should be False)")

Call 1 negatives: [ 359135  167878  938490 1739123]
Call 2 negatives: [ 914866  228372 1496917  802848]
Identical? False   (should be False)


In [7]:
# Exclusion correctness: sampled negatives must never collide with the user's rated books.
# Sample 100 random positives, check each one's negatives against the user's exclusion set.
n_checks = 100
rng = np.random.default_rng(0)
violations = 0
for idx in rng.integers(0, len(dataset), size=n_checks):
    u, _, negs = dataset[int(idx)]
    excluded = dataset.exclude[u]
    overlap = np.intersect1d(negs, excluded)
    if len(overlap) > 0:
        violations += 1
        print(f"FAIL idx={idx} user={u}: overlap={overlap}")
print(f"\nChecked {n_checks} samples, {violations} violations")
assert violations == 0, "Negative sampler leaked a rated book"


Checked 100 samples, 0 violations


In [8]:
# Preview of how the model will consume this. At training time:
#   loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=4)
#   for user_idx, pos_book_idx, neg_book_idxs in loader:
#       user_idx        shape (256,)
#       pos_book_idx    shape (256,)
#       neg_book_idxs   shape (256, 4)
#   ...the user tower fetches user_features[user_idx], the item tower fetches item features for both
#   the positive and the 4 negatives, computes dot products, applies BCEWithLogitsLoss with labels
#   [1, 0, 0, 0, 0] per anchor. Resampling-per-call means each epoch sees fresh negatives automatically.
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=4, shuffle=False)
batch = next(iter(loader))
u_b, pos_b, neg_b = batch
print(f"user_idx       shape={tuple(u_b.shape)}   dtype={u_b.dtype}")
print(f"pos_book_idx   shape={tuple(pos_b.shape)}   dtype={pos_b.dtype}")
print(f"neg_book_idxs  shape={tuple(neg_b.shape)}   dtype={neg_b.dtype}")

user_idx       shape=(4,)   dtype=torch.int64
pos_book_idx   shape=(4,)   dtype=torch.int64
neg_book_idxs  shape=(4, 4)   dtype=torch.int64
